# Demo 7: Task span vs code-progress span: which one holds the next sub-goal?

**Question.** By the pivot, the model has seen two things that could tell it what to write next: the **task span**
(the numbered list of sub-tasks in the instruction) and the **code-progress span** (the code written so far, from the
end of the user turn up to the pivot). The progress span's latents may already hold a "pre-committed" decision about the
next line. Which span does the pivot actually read the sub-goal from?

**Setup.** As in Demo 5, two Python prompts that share the progress and differ only in sub-task (3):

```
p_src:  (1) a  (2) b  (3) c      -> writes c
p_dst:  (1) a  (2) b  (3) d      -> writes d
```

The code-progress span is *token-identical* in both prompts, but its latents are not: in SRC they were computed while
attending to a task list that ends in c. So we can transplant each span from SRC into DST separately, a 2 × 2 grid:

| | progress from DST | progress from SRC |
|---|---|---|
| **task from DST** | clean DST (no-op) | progress only |
| **task from SRC** | task only | task + progress |

**Two ways to transplant.** (i) *Residual*: the span's input embeddings and every layer's output ← SRC, every other
position up to the pivot pinned to clean DST. (ii) *KV through the goal heads*: only the 250 goal heads, from the pivot
on, read SRC's keys / values over the span (values × 4), as in Demo 5. The pivot itself always computes freely.

**Paper.** §5.2 (`fig:task-vs-prog`), App. `app:pre-com` (`fig:pre-com-kv`).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Task span or code progress: which one carries the goal? The 2×2 transfer grid (App. J).](figures/07_task_vs_progress_setup.png)

*Task span or code progress: which one carries the goal? The 2×2 transfer grid (App. J).*

![The (source-task, destination-progress) cell: residual patching of the task span, with the code progress restored from the destination.](figures/07_task_vs_progress_residual.png)

*The (source-task, destination-progress) cell: residual patching of the task span, with the code progress restored from the destination.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
Defaults of `scripts/nrgs/task_vs_progress_eval.py`, the paper's configuration: Python→Python, instances from
`sample_multi_instances(seed=1123)`, greedy 64 tokens, no GoalScope readout, `goal250` = the 250 goal heads.
We run the two methods the paper reports: residual (`fig:task-vs-prog`, right panel) and KV on goal-250 at V × 4 (the
boxed cell of `fig:pre-com-kv`).

**Why this pair.** `int`, shared `(1) digit_sum (2) is_perfect_square`, SRC `(3) count_set_bits` vs DST
`(3) is_prime`: the two sub-goals look nothing alike (`bin(n).count('1')` vs a primality loop), so every cell is
unambiguous. In the reported run this instance shows the pattern of the pooled grid under both methods (task → c,
progress → d, task + progress → c).

In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
DATATYPE, SRC_LANG, DST_LANG = 'int', 'python', 'python'
INST_KEY = 'multi|int|shared=digit_sum-is_perfect_square|src=count_set_bits|dst=is_prime'
SEED, MAX_NEW = 1123, 64
METHODS  = {'residual': dict(method='residual', heads=None, v=1.0),          # (method, heads, V) of TP.CONFIGS
            'KV · goal 250 · V×4': dict(method='kv', heads='goal250', v=4.0)}

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model

In [3]:
import json, collections
import torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
import src.nrgs.task_vs_progress as TP
from src.utils import env_utils

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
mt = ctx.mt
goal = [tuple(h) for h in json.load(open(G250))]
headsets = {'goal250': goal, 'all': [(l, h) for l in range(mt.n_layer) for h in range(mt.n_heads)]}
print(f'goal heads: {len(goal)} of {mt.n_layer * mt.n_heads}')

2026-10-30 21:38:27 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 of 2560


## 1. The pair and its two spans
`TP.geometry` builds both prompts (`PP.build_geometry`, task span `\n`-padded at the front so SRC and DST windows align
token for token) and the progress window `<|im_end|> .. {var}` (aligned from the end). Below: the **task span** in
cyan (SRC) / magenta (DST), the **progress span** in yellow, the pivot in red.

In [4]:
insts, _ = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=32, seed=SEED)
inst = next(i for i in insts if PP.inst_key(i) == INST_KEY)
IDX = insts.index(inst)
G = TP.geometry(inst, ctx=ctx)
print(f'instance #{IDX}: {INST_KEY}')
print('spans (DST positions): task', G['W']['task'][1][0], '..', G['W']['task'][1][-1],
      '| progress', G['W']['progress'][1][0], '..', G['W']['progress'][1][-1], '| pivot', G['piv_d'])

def show(side, color):
    tok = G[f'{side}_tok']; ids = tok['input_ids'][0].tolist()
    task, prog = (set(G['W'][s][0 if side == 'src' else 1]) for s in ('task', 'progress'))
    start = min(task) - 1
    while start > 0 and ids[start] != mt.tokenizer.convert_tokens_to_ids('<|im_start|>'):
        start -= 1
    txt = Text()
    for i in range(start, len(ids)):
        st = (f'bold {color} on #f1f5f9' if i in task else 'bold #b45309 on #fef3c7' if i in prog
              else 'bold white on red' if i == len(ids) - 1 else 'dim')
        txt.append(mt.tokenizer.decode([ids[i]]), style=st)
    console.rule(f'{side.upper()} prompt'); console.print(txt)

show('src', 'cyan'); show('dst', 'magenta')
s_ids, d_ids = PP._ids(G['src_tok']), PP._ids(G['dst_tok'])
print('progress tokens identical in SRC and DST:',
      [s_ids[q] for q in G['W']['progress'][0]] == [d_ids[q] for q in G['W']['progress'][1]])

instance #11: multi|int|shared=digit_sum-is_perfect_square|src=count_set_bits|dst=is_prime
spans (DST positions): task 61 .. 93 | progress 94 .. 142 | pivot 143


──────────────────────────────────────────────────────────────── SRC prompt ────────────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) the sum of the digits
(2) whether the number is a perfect square
(3) the number of set bits in the binary representation
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = sum(int(d) for d in str(abs(n)))
    b = n >= 0 and int(n**0.5)**2 == n
    c =

──────────────────────────────────────────────────────────────── DST prompt ────────────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:




(1) the sum of the digits
(2) whether the number is a perfect square
(3) whether the number is prime
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = sum(int(d) for d in str(abs(n)))
    b = n >= 0 and int(n**0.5)**2 == n
    c =

progress tokens identical in SRC and DST: True


## 2. Clean runs

In [5]:
clean = PP.clean_runs(dict(src_tok=G['src_tok'], dst_tok=G['dst_tok']), readout_heads=None, max_new=MAX_NEW,
                      readout=False, ctx=ctx)
first = lambda s: s.strip().splitlines()[0] if s.strip() else ''
console.print(f'SRC writes c = [cyan]{escape(first(clean["gen_src"]))}[/]   ({escape(inst.expected_next_desc)})')
console.print(f'DST writes d = [magenta]{escape(first(clean["gen_dst"]))}[/]   ({escape(inst.dst_next_desc)})')

SRC writes c = bin(n).count('1')   (the number of set bits in the binary representation)

DST writes d = n > 1 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))   (whether the number is prime)

## 3. The 2 × 2 grid, twice
`TP.make_stack` returns the intervention for one (method, span); `TP.run_patched` decodes DST under it;
`TP.judge` asks the batch eval's oracle whether the continuation's first computation implements c (SRC's sub-goal,
label `src`) or d (DST's own, `dst`). Claude calls are cached on disk.

In [6]:
SPANS = {('dst', 'dst'): 'none', ('src', 'dst'): 'task', ('dst', 'src'): 'progress', ('src', 'src'): 'task+progress'}
grid = {}
for mname, m in METHODS.items():
    for cell, span in SPANS.items():
        if span == 'none':
            gen = clean['gen_dst']
        else:
            stack = TP.make_stack(G, m['method'], span, m['heads'], m['v'], headsets=headsets, ctx=ctx)
            gen = TP.run_patched(G, stack, max_new=MAX_NEW, ctx=ctx)
        grid[mname, span] = dict(gen=gen, causal=TP.judge(G, clean, gen))

LAB = {'src': '[bold cyan]c (SRC)[/]', 'dst': '[bold magenta]d (DST)[/]'}
for mname in METHODS:
    t = Table(title=f'{mname}: what DST writes at the pivot', show_lines=True)
    t.add_column(''); t.add_column('progress span from DST', max_width=52); t.add_column('progress span from SRC', max_width=52)
    for ts in ('dst', 'src'):
        row = []
        for ps in ('dst', 'src'):
            r = grid[mname, SPANS[ts, ps]]
            row.append(f"{LAB.get(r['causal']['label'], r['causal']['label'])}\n{escape(first(r['gen']))}")
        t.add_row(f'task span from {ts.upper()}', *row)
    console.print(t)

                                               residual: what DST writes at the pivot                                               
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                    ┃ progress span from DST                               ┃ progress span from SRC                               ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ task span from DST │ d (DST)                                              │ d (DST)                                              │
│                    │ n > 1 and all(n % i != 0 for i in range(2,           │ n > 1 and all(n % i != 0 for i in range(2,           │
│                    │ int(n**0.5) + 1))                                    │ int(n**0.5) + 1))                                    │
├────────────────────┼──────────────────────────────────────────────────────┼──────────────────────────────────────────────────────┤
│ task span from SRC │ c (SRC)                                              │ c (SRC)                                              │
│                    │ bin(n).count('1') if n >= 0 else bin(-n).count('1')  │ bin(n).count('1')                                    │
└────────────────────┴──────────────────────────────────────────────────────┴──────────────────────────────────────────────────────┘

                                         KV · goal 250 · V×4: what DST writes at the pivot                                          
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                    ┃ progress span from DST                               ┃ progress span from SRC                               ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ task span from DST │ d (DST)                                              │ d (DST)                                              │
│                    │ n > 1 and all(n % i != 0 for i in range(2,           │ n > 1 and all(n % i != 0 for i in range(2,           │
│                    │ int(n**0.5) + 1))                                    │ int(n**0.5) + 1))                                    │
├────────────────────┼──────────────────────────────────────────────────────┼──────────────────────────────────────────────────────┤
│ task span from SRC │ c (SRC)                                              │ c (SRC)                                              │
│                    │ bin(n).count('1')                                    │ bin(n).count('1') if n >= 0 else bin(n &             │
│                    │                                                      │ -n).count('1')                                       │
└────────────────────┴──────────────────────────────────────────────────────┴──────────────────────────────────────────────────────┘

Both methods give the same picture. Transplanting the **task span** alone flips DST to c (`bin(n).count('1')`);
transplanting the **progress span** alone leaves DST writing its own primality check d, even though in SRC those
latents were computed while attending to a list ending in c. Task + progress behaves like task alone. So whatever
"pre-commitment" the progress latents might carry, the pivot does not read the next sub-goal from them: it re-reads
it from the task span, and through the goal heads' keys / values alone (KV, goal 250) that is enough.

## 4. Cross-check against the reported run
The cached sweep entry of this instance (`results/notebook_caches/task_vs_progress/<model>/int__python-python__seed1123__mx64__ro0__task+gap/`),
then the reported causal-shift rates of the same cells, mean over the 4 datatypes (n = 32 each).

In [7]:
root = env_utils.DEFAULT_RESULTS_DIR
cdir = TP.cache_dir(root, MODEL.split('/')[-1], DATATYPE, SRC_LANG, DST_LANG, seed=SEED, max_new=MAX_NEW)
entry = json.load(open(TP.cache_path(cdir, inst)))
summ = {dt: json.load(open(TP.cache_dir(root, MODEL.split('/')[-1], dt, SRC_LANG, DST_LANG, seed=SEED, max_new=MAX_NEW)
                           + '.summary.json')) for dt in ('int', 'str', 'list[int]', 'list[str]')}
t = Table(title='this notebook vs. the reported run')
for c in ('method', 'span', 'paper verdict', 'ours', 'same gen?', 'int rate', 'mean over 4 datatypes'):
    t.add_column(c)
for mname, m in METHODS.items():
    for span in ('none', 'task', 'progress', 'task+progress'):
        name = TP.cfg_name('none', None, 'none', 1.0) if span == 'none' else TP.cfg_name(m['method'], m['heads'], span, m['v'])
        row = entry['configs'][name]
        ours = grid[mname, span]
        rates = [summ[dt][name]['causal_shift'] for dt in summ]
        t.add_row(mname, span, row['causal'], ours['causal']['label'], str(row['gen'] == ours['gen']),
                  f"{summ['int'][name]['causal_shift']:.3f}", f'{sum(rates) / len(rates):.3f}')
console.print(t)

                                     this notebook vs. the reported run                                      
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ method              ┃ span          ┃ paper verdict ┃ ours ┃ same gen? ┃ int rate ┃ mean over 4 datatypes ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ residual            │ none          │ dst           │ dst  │ True      │ 0.000    │ 0.000                 │
│ residual            │ task          │ src           │ src  │ True      │ 0.781    │ 0.891                 │
│ residual            │ progress      │ dst           │ dst  │ True      │ 0.062    │ 0.062                 │
│ residual            │ task+progress │ src           │ src  │ True      │ 0.969    │ 0.984                 │
│ KV · goal 250 · V×4 │ none          │ dst           │ dst  │ True      │ 0.000    │ 0.000                 │
│ KV · goal 250 · V×4 │ task          │ src           │ src  │ True      │ 0.969    │ 0.969                 │
│ KV · goal 250 · V×4 │ progress      │ dst           │ dst  │ True      │ 0.344    │ 0.258                 │
│ KV · goal 250 · V×4 │ task+progress │ src           │ src  │ True      │ 0.969    │ 0.984                 │
└─────────────────────┴───────────────┴───────────────┴──────┴───────────┴──────────┴───────────────────────┘

All eight cells reproduce the cached sweep entry token for token (same generation, same verdict). The instance is
typical of the pooled rates: task 0.891 (residual) / 0.969 (KV) and task + progress 0.984 shift DST to c, while
progress alone shifts it only 0.062 (residual) / 0.258 (KV V × 4) of the time.

## Where this goes in the paper / how to reproduce at scale
- **Commands**: `job_list/diff_eval/task_vs_progress_32b.json` and `task_vs_progress_32b_v2.json` (one job per datatype),
  e.g. `python scripts/nrgs/task_vs_progress_eval.py --datatypes int`; smoke `--max-instances 2`. The sweep runs all of
  `TP.CONFIGS` (none + KV on {goal-250, all heads} × {task, progress, task+progress} × V {1, 2, 4} + residual × 3 spans)
  and caches per instance in `results/notebook_caches/task_vs_progress/...__task+gap/` (resumable); point `RESULTS_DIR`
  in `env.yml` to a scratch dir for a re-run.
- **Paper numbers** (mean over 4 datatypes): residual task 0.891 / progress 0.062 / task+progress 0.984; KV goal-250
  task V × 4 0.969, progress V × 4 0.258.
- **Generator**: `notebooks/fig_n_table_generators/task_vs_progress.ipynb` (cell 5 `tvp_residual_grid`, cell 6
  `tvp_kv_grids`).
- **Related**: KV transfer of the task span (Demo 5), Q transfer (Demo 6).